#### Load data & libraries

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

In [2]:
# 1. Khởi tạo cấu hình kết nối SparkSession
spark = SparkSession.builder \
    .appName("Feature_Transformation") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# 2. Đọc dữ liệu trực tiếp từ HDFS
hdfs_input_path = "hdfs://localhost:9000/big_data/hotel_bookings_feature_extracted.csv"
df = spark.read.csv(hdfs_input_path, header=True, inferSchema=True)

In [7]:
df = spark.read.csv("../data/hotel_bookings_feature_extracted.csv", header = True, inferSchema=True)

In [8]:
print(f'Số cột: {len(df.columns)}')
print(f'Tên cột: {df.columns}')

Số cột: 21
Tên cột: ['hotel', 'is_canceled', 'lead_time', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'reserved_room_type', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights', 'total_estimated_cost', 'is_family', 'total_past_bookings', 'cancellation_rate']


In [9]:
# Xác định Cột Mục Tiêu
target_col = "is_canceled"

# 1. Nhóm biến Phân loại (Chữ) -> Cần dùng StringIndexer & OneHotEncoder
categorical_cols = [
    "hotel",
    "meal",
    "country",
    "market_segment", "distribution_channel", 
    "reserved_room_type",
    "deposit_type",
    "customer_type",
    "agent"
]

# 2. Nhóm biến Liên tục (Số) -> Cần dùng StandardScaler
numeric_cols = [
    "lead_time",
    "total_guests", "total_nights", "total_estimated_cost", 
    "total_past_bookings", "cancellation_rate", "days_in_waiting_list", 
    "required_car_parking_spaces", "total_of_special_requests"
]

# 3. Nhóm biến Cờ (Đã là 0/1 sẵn)
flag_cols = [
    "is_repeated_guest", "is_family"
]

#### Stratified Train-Test Split

In [7]:
# Tạo cột ID duy nhất không gây thắt cổ chai hệ thống
df = df.withColumn("row_id", F.expr("uuid()"))

# Lấy mẫu 80% cho từng lớp để bảo toàn tỷ lệ Imbalance ban đầu
fractions = {0: 0.8, 1: 0.8}
train_data = df.sampleBy(target_col, fractions=fractions, seed=1)

# Tập Test là phần còn lại thông qua Anti-Join
test_data = df.join(train_data, on="row_id", how="left_anti")

# Xóa cột ID đi để không đưa vào mô hình học
train_data = train_data.drop("row_id")
test_data = test_data.drop("row_id")

print(f"Số dòng Train: {train_data.count()} | Test: {test_data.count()}")

Số dòng Train: 94821 | Test: 23500


In [8]:
train_total = train_data.count()
test_total = test_data.count()

train_dist = (
    train_data
    .groupBy(target_col)
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") * 100.0 / train_total, 2))
    .withColumn("dataset", F.lit("Train"))
)

test_dist = (
    test_data
    .groupBy(target_col)
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") * 100.0 / test_total, 2))
    .withColumn("dataset", F.lit("Test"))
)

(train_dist
 .unionByName(test_dist)
 .select("dataset", target_col, "count", "percentage")
 .orderBy(target_col, "dataset")
 .show())

+-------+-----------+-----+----------+
|dataset|is_canceled|count|percentage|
+-------+-----------+-----+----------+
|   Test|          0|14854|     63.21|
|  Train|          0|59393|     62.64|
|   Test|          1| 8646|     36.79|
|  Train|          1|35428|     37.36|
+-------+-----------+-----+----------+



#### Pipeline

In [9]:
stages = []

# 5.1. Xử lý biến Categorical
for cat_col in categorical_cols:
    indexer = StringIndexer(inputCol=cat_col, outputCol=f"{cat_col}_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=f"{cat_col}_idx", outputCol=f"{cat_col}_vec")
    stages += [indexer, encoder]

# 5.2. Chuẩn hóa biến Numeric
num_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="numeric_features_vec")
scaler = StandardScaler(inputCol="numeric_features_vec", outputCol="scaled_numeric_features", withStd=True, withMean=True)
stages += [num_assembler, scaler]

# 5.3. Gom tất cả thành 1 Vector 'features' duy nhất
assembler_inputs = [f"{c}_vec" for c in categorical_cols] + ["scaled_numeric_features"] + flag_cols
final_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages += [final_assembler]

pipeline = Pipeline(stages=stages)
print("Đã xây xong Pipeline tiền xử lý")

Đã xây xong Pipeline tiền xử lý


#### Pipeline fit and transform

In [10]:
# Fit pipeline trên tập Train
print("Đang Fit Pipeline trên tập Train...")
pipeline_model = pipeline.fit(train_data)

# Transform trên cả Train và Test
print("Đang Transform Pipeline trên tập Train")
train_vectorized = pipeline_model.transform(train_data).select("features", F.col(target_col).alias("label"))

print("Đang Transform Pipeline trên tập Test")
test_vectorized = pipeline_model.transform(test_data).select("features", F.col(target_col).alias("label"))

print("Đã hoàn thành Vector hóa dữ liệu Train và Test")

Đang Fit Pipeline trên tập Train...
Đang Transform Pipeline trên tập Train
Đang Transform Pipeline trên tập Test
Đã hoàn thành Vector hóa dữ liệu Train và Test


In [11]:
# Lấy kế hoạch thực thi
print("\n--- KẾ HOẠCH THỰC THI (PHYSICAL PLAN) ---")
train_vectorized.explain()


--- KẾ HOẠCH THỰC THI (PHYSICAL PLAN) ---
== Physical Plan ==
*(1) Project [features#1008, is_canceled#18 AS label#1009]
+- *(1) Project [is_canceled#18, UDF(struct(meal_vec, meal_vec#838, country_vec, country_vec#856, market_segment_vec, market_segment_vec#874, distribution_channel_vec, distribution_channel_vec#892, reserved_room_type_vec, reserved_room_type_vec#910, deposit_type_vec, deposit_type_vec#928, customer_type_vec, customer_type_vec#946, agent_vec, agent_vec#964, hotel_vec, hotel_vec#982, scaled_numeric_features, UDF(numeric_features_vec#997), is_repeated_guest_double_VectorAssembler_a27441e96908, cast(is_repeated_guest#33 as double), is_family_double_VectorAssembler_a27441e96908, cast(is_family#49 as double))) AS features#1008]
   +- *(1) Project [is_canceled#18, is_repeated_guest#33, is_family#49, meal_vec#838, country_vec#856, market_segment_vec#874, distribution_channel_vec#892, reserved_room_type_vec#910, deposit_type_vec#928, customer_type_vec#946, agent_vec#964, UDF(

#### Train-Test split & Data export

In [12]:
# Lưu Pipeline Model
pipeline_model.write().overwrite().save("./hotel_preprocessing_pipeline_model")

# Lưu Train/Test Data dạng Parquet
train_vectorized.write.mode("overwrite").parquet("./hotel_train_vectorized.parquet")
test_vectorized.write.mode("overwrite").parquet("./hotel_test_vectorized.parquet")

print("Xuất dữ liệu và pipeline hoàn tất")

Xuất dữ liệu và pipeline hoàn tất


#### Output check

In [13]:
from pyspark.ml import PipelineModel
print("=== 1. KIỂM THỬ LOAD DỮ LIỆU PARQUET ===")
try:
    # Thử load folder mà bạn vừa lưu
    test_load_train = spark.read.parquet("./hotel_train_vectorized.parquet")
    
    # Check 1: In ra số lượng dòng
    print(f"[OK] Đã load thành công! Số dòng: {test_load_train.count()}")
    
    # Check 2: Kiểm tra Schema xem cấu trúc Vector có được bảo toàn không
    print("[OK] Cấu trúc Schema hiện tại (Nên chỉ có 2 cột: features và label):")
    test_load_train.printSchema()
    
    # Check 3: Nhìn thử 2 dòng dữ liệu
    print("[OK] Dữ liệu mẫu (Nhìn xem cột features có dạng SparseVector không):")
    test_load_train.show(2, truncate=False)
except Exception as e:
    print(f"[LỖI] Dữ liệu Parquet có vấn đề: {e}")


print("\n=== 2. KIỂM THỬ LOAD PIPELINE MODEL ===")
try:
    # Thử load folder Model
    test_load_pipeline = PipelineModel.load("./hotel_preprocessing_pipeline_model")
    
    # In ra các trạm (stages) bên trong ống nước để chứng minh nó đã nhớ cấu trúc
    print("[OK] Load Pipeline thành công! Các trạm xử lý bên trong gồm:")
    for i, stage in enumerate(test_load_pipeline.stages):
        print(f"  - Trạm {i+1}: {stage.__class__.__name__} ({stage.uid})")
except Exception as e:
    print(f"[LỖI] Pipeline Model có vấn đề: {e}")

=== 1. KIỂM THỬ LOAD DỮ LIỆU PARQUET ===
[OK] Đã load thành công! Số dòng: 94821
[OK] Cấu trúc Schema hiện tại (Nên chỉ có 2 cột: features và label):
root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)

[OK] Dữ liệu mẫu (Nhìn xem cột features có dạng SparseVector không):
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                              |label|
+------------------------------------------------------------------------------------------

#### Reproduce

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel

In [ ]:
# 1. Khởi tạo cấu hình kết nối SparkSession
spark = SparkSession.builder \
    .appName("Feature_Transformation_reproduce") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# 1. Load pipeline đã lưu
loaded_pipeline = PipelineModel.load("../transform_model/hotel_preprocessing_pipeline_model")
df = spark.read.csv("hdfs://localhost:9000/big_data/hotel_bookings_feature_extracted.csv", header = True, inferSchema=True)

# 2. Đổ dữ liệu chưa qua xử lý vào pipeline
transformed_df = loaded_pipeline.transform(df)

#### Feature mapping file

In [9]:
import pandas as pd
import re

# Lấy mảng thuộc tính (metadata) bên trong cột "features" của Spark ra
attrs = transformed_df.schema["features"].metadata["ml_attr"]["attrs"]

feature_mapping = []

for attr_type in attrs:
    feature_mapping.extend(attrs[attr_type])

# Sắp xếp danh sách dựa trên chỉ số Index (0, 1, 2...) để mô hình đọc đúng thứ tự
feature_mapping = sorted(
    feature_mapping,
    key=lambda x: x["idx"]
)

# Đẩy dữ liệu vào bảng Pandas để dễ thao tác
mapping_pd = pd.DataFrame([
    {
        "feature_index": x["idx"],
        "feature_name": x["name"]
    }
    for x in feature_mapping
])

numeric_cols = [
    "lead_time",
    "total_guests", "total_nights", "total_estimated_cost", 
    "total_past_bookings", "cancellation_rate", "days_in_waiting_list", 
    "required_car_parking_spaces", "total_of_special_requests"
]
# LƯU Ý: numeric_cols phải là danh sách giống hệt lúc tạo Pipeline
def map_real_numeric_names(feature_name):
    """
    Hàm dò tìm và dán nhãn lại tên cho các biến số bị Spark làm mất tên.
    """
    match = re.match(r"scaled_numeric_features_(\d+)", feature_name)
    if match:
        idx = int(match.group(1))
        if idx < len(numeric_cols):
            return numeric_cols[idx]
    return feature_name

# Áp dụng máy quét đè lên toàn bộ cột tên
mapping_pd["feature_name"] = mapping_pd["feature_name"].apply(map_real_numeric_names)

# Xuất file CSV
mapping_pd.to_csv("../transform_model/feature_mapping.csv", index=False, encoding="utf-8-sig")

print("Đã hoàn tất trích xuất và sửa tên đặc trưng!")
print("\n--- XEM TRƯỚC 10 DÒNG ĐẦU ---")
print(mapping_pd.head(10))

Đã hoàn tất trích xuất và sửa tên đặc trưng!

--- XEM TRƯỚC 10 DÒNG ĐẦU ---
   feature_index        feature_name
0              0         meal_vec_BB
1              1         meal_vec_HB
2              2         meal_vec_SC
3              3  meal_vec_Undefined
4              4         meal_vec_FB
5              5     country_vec_PRT
6              6     country_vec_GBR
7              7     country_vec_FRA
8              8     country_vec_ESP
9              9     country_vec_DEU


In [14]:
spark.stop()